<a href="https://colab.research.google.com/github/UdayAnalyst/A01_5512/blob/main/Module3/Lab4_Model_In_Production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Lab 4 · Deploying Your Model to Production
### Fit a really good model in Colab. Deploy it to Google Cloud. Watch it price every new car.

*Solo lab · about 2 hours · Colab · Dr. Wanik's cleaned Hartford cars, then your own Google Cloud bucket*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/Lab4_Model_In_Production.ipynb)

Since Lab 3 your pipeline scrapes cars, extracts them with **your** regex, and builds a clean master table every hour.
Tonight you **fit a really good model** (tuned, and better than the baseline) and **deploy it** into that pipeline, so it
prices every new car on its own, every hour. SHAP comes along for the ride: production also saves **why** it gave each price.

| Part | Time | What you do |
|---|---|---|
| **0. Before you start** | 5 min | Get `predict-shap` deploying (Sync fork, Run workflow, or Cloud Shell) while you work |
| **1. Load** | 10 min | Dr. Wanik's cleaned `listings_master.csv` (453 Hartford cars, Sep 28 to Oct 13, after the A06 regex fix) |
| **2. Clean** | 10 min | The last mess (`Ford` vs `ford`, `chevy` vs `chevrolet`), then flag weird prices, miles and years |
| **3. Baseline** | 10 min | Two numbers to beat: guess the median, and the plain decision tree your pipeline already trains |
| **4. Tune** | 25 min | Tune a decision tree, a random forest and KNN with cross-validation; the best one becomes your champion |
| **Your turn** | 15 min | **You code it:** actual-vs-predicted scatter, error histogram and summary stats, champion vs baseline |
| **5. Explain** | 20 min | SHAP on your champion: the global view (bar + beeswarm), one car, then **price a car you know** |
| **6. Ship** | 20 min | Save the model, upload it to your bucket (account checked first), and run `predict-shap` from Colab |
| **7. Your model at work** | 5 min | Your frozen model's prices and SHAP values for brand-new cars, in Colab and in Google Cloud |
| **8. Wrap up** | 5 min | Five short answers, submit |

**First: File → Save a copy in Drive.** This notebook opens from GitHub, so your edits disappear when you close the tab unless you work in your own copy.

**Stuck?** Ask Claude: paste the cell, the full error, and what you expected.

## The big picture
Your pipeline already scrapes (:00), extracts (:10 regex, :12 Gemini), combines (:15) and trains a basic tree (:20).
Tonight adds one more hourly step, **:25 · predict-shap**: it loads **your** model, predicts every new car, and saves
**why** (SHAP) next to **what** (the prediction).

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-labs/master/Module3/img/OPIM5512_M3_Pipeline_Overview.png" alt="How your pipeline works" width="100%">

# Part 0 · Before you start: get `predict-shap` running (5 min)

Dr. Wanik added a new Cloud Function, **`predict-shap`**, to the course repo. It takes about 3 minutes to deploy, so
start it **now** and let it finish while you work. Open **your** myscrapers repo on GitHub and find your case:

| What you see on your repo's Code tab | Your case |
|---|---|
| **"This branch is N commits behind drdave-teaching/myscrapers"** | **A**: Sync fork |
| "forked from drdave-teaching/myscrapers", **0 behind**, and a `cloud_function/predict-shap` folder | **B**: run the workflow once |
| **No** "forked from drdave-teaching/myscrapers" under the repo name (you imported, cloned, or copied it) | **C**: Cloud Shell |

**Case A: Sync fork** (the same steps as the Sync-fork video)
1. Click **Sync fork**. GitHub offers two buttons. **Click "Update branch."**
   **Never click "Discard commits":** the commits it would discard are **your A06 regex fix**, and "Contact
   Information" would come back. "Update branch" keeps your fix **and** adds `predict-shap` on top.
2. Afterwards your repo says it is **ahead** (your A06 work) and **0 behind**. Good.
3. **Actions** tab: **Deploy predict-shap** starts by itself. No run after a minute? Do Case B.

**Case B: run the workflow once.** You forked after `predict-shap` was added, so the files are there but it was never
deployed. **Actions** tab → **Deploy predict-shap** (left list) → **Run workflow** → **Run workflow**.

**Case C: no fork? Deploy it from Cloud Shell.** Your repo is not linked to the course repo, so Sync fork can never
bring you updates. This block deploys `predict-shap` straight from Dr. Wanik's repo, with exactly the settings the
GitHub workflow uses. Open **Cloud Shell**, change the first two lines, and paste the whole block:
```bash
PROJECT_ID="YOUR-PROJECT-ID"
BUCKET_NAME="YOUR-PROJECT-ID"
REGION="us-central1"
gcloud config set project "$PROJECT_ID"
RUNTIME_SA="cf-runtime@${PROJECT_ID}.iam.gserviceaccount.com"

mkdir -p ~/predict-shap && cd ~/predict-shap
curl -sSLO https://raw.githubusercontent.com/drdave-teaching/myscrapers/main/cloud_function/predict-shap/main.py
curl -sSLO https://raw.githubusercontent.com/drdave-teaching/myscrapers/main/cloud_function/predict-shap/requirements.txt

gcloud functions deploy predict-shap --gen2 --region="$REGION" --runtime=python312 --source=.   --entry-point=predict_shap_http --trigger-http --no-allow-unauthenticated   --timeout=900 --memory=2Gi --service-account="$RUNTIME_SA"   --set-env-vars="PROJECT_ID=$PROJECT_ID,GCS_BUCKET=$BUCKET_NAME,DATA_KEY=structured/datasets/listings_master.csv,OUTPUT_PREFIX=structured/preds_lab4,MODEL_PREFIX=models"

gcloud functions add-invoker-policy-binding predict-shap --gen2 --region="$REGION"   --member="serviceAccount:$RUNTIME_SA"
URL=$(gcloud functions describe predict-shap --gen2 --region="$REGION" --format='value(serviceConfig.uri)')
gcloud scheduler jobs create http predict-shap-hourly --location="$REGION"   --schedule="25 * * * *" --time-zone="America/New_York" --uri="$URL" --http-method=POST   --message-body='{"dry_run":false}' --attempt-deadline=900s   --oidc-service-account-email="$RUNTIME_SA" --oidc-token-audience="$URL"
```
The deploy takes about 3 minutes. If GCP asks to enable an API, type `y`. If the last line says the job already
exists, you are fine. In Case C, when Dr. Wanik updates `predict-shap` later, paste this block again (and change the
last command's `create` to `update`).

Until you upload a model, `predict-shap` politely does nothing (it reports `"status": "no_model"`).

# Part 1 · Load (10 min)

## 1a. Match production's library versions
Your model has to load inside a Google Cloud Function later, and a model saved with one version of scikit-learn may not
load in another. So we install the **exact** versions `predict-shap` uses. Production pins them in
[`cloud_function/predict-shap/requirements.txt`](https://github.com/drdave-teaching/myscrapers/blob/main/cloud_function/predict-shap/requirements.txt)
(`scikit-learn==1.6.1`, `shap==0.48.0`): if you ever change one, change it in both places. (If Colab asks you to **restart the
session**, click it, then run this notebook again from the top.)

In [ ]:
!pip install -q scikit-learn==1.6.1 shap==0.48.0

import sklearn
import shap
print("scikit-learn", sklearn.__version__, "| shap", shap.__version__, "(production uses 1.6.1 and 0.48.0)")

## 1b. Pick your data
- **Start here (everyone, tonight):** leave `USE_MY_OWN_DATA = False`. You get the real `listings_master.csv` from
  Dr. Wanik's pipeline **after** the A06 fix: 453 Hartford cars scraped Sep 28 to Oct 13, make and model pulled by the Lab 3 regex. No sign-in.
  Everyone trains on the same cars, so everyone's numbers are comparable.
- **Your own cars (later, for a custom model):** set `USE_MY_OWN_DATA = True` and re-run from here. You need your A06
  regex fix shipped (otherwise half your makes say `contact`) and ideally 100+ cars. The next cell checks both for you.

Type your project ID and bucket either way: Part 6 uses them to ship your model.

In [ ]:
USE_MY_OWN_DATA = False            # True = train on YOUR bucket's listings_master.csv
PROJECT_ID = "YOUR-PROJECT-ID"     # e.g. "myscrapers-dww05002-f26-510015"
BUCKET_NAME = "YOUR-PROJECT-ID"    # usually the same as your project ID

In [ ]:
import io
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module3/data/lab4_listings_master_snapshot.csv"

if USE_MY_OWN_DATA:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    my_bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    # Your bucket is private, so pandas cannot just open "gs://..." like a public web address.
    # Instead the storage client (signed in as you) downloads the file's raw contents into memory...
    csv_bytes = my_bucket.blob("structured/datasets/listings_master.csv").download_as_bytes()
    # ...and io.BytesIO wraps those bytes so pandas can read them as if they were a file on disk (nothing is saved).
    raw = pd.read_csv(io.BytesIO(csv_bytes), dtype={"post_id": str})
    print("Loaded YOUR listings_master.csv from gs://" + BUCKET_NAME)
    contact_rows = int((raw["make"].astype(str).str.lower() == "contact").sum())
    if contact_rows > 0:
        print("WARNING:", contact_rows, "cars still say make = 'contact'. Ship your A06 regex fix (and re-extract) first.")
    if len(raw) < 100:
        print("WARNING: only", len(raw), "cars. A custom model needs more data: let your pipeline run a few more days.")
else:
    raw = pd.read_csv(DATA_URL, dtype={"post_id": str})
    print("Loaded Dr. Wanik's cleaned Hartford snapshot")

print(raw.shape[0], "cars,", raw.shape[1], "columns")
raw.head()

# Part 2 · Clean (10 min)

The regex fixed `Contact Information`, but real data always has a little mess left. Look at the make counts:
`ford` and `Ford` are the same car, and so are `chevy`, `chevrolet` and `Chevrolet`. A model would treat them as
different brands.

**Important:** `predict-shap` runs **this exact cleaning** on every new car before predicting. If you change it
here, your model and production stop agreeing.

In [ ]:
print(raw["make"].value_counts().head(12))

In [ ]:
FEATURES = ["year", "mileage", "make", "model"]
NICKNAMES = {"chevy": "chevrolet", "vw": "volkswagen", "mercedes": "mercedes-benz", "benz": "mercedes-benz"}


def clean_cars(df):
    out = df.copy()                       # work on a copy, never change the raw data

    # NUMBERS: strip everything that is not a digit or a period, then convert to a number.
    # The regex [^\d.]+ means: [^ ... ] "any character that is NOT" a digit (\d) or a period (.), and the + means
    # "one or more of them in a row". Each run is replaced with nothing:  "$12,500" -> "12500"   "120,000 mi" -> "120000"
    # errors="coerce" turns anything still unreadable into NaN (blank) instead of crashing.
    for col in ["price", "year", "mileage"]:
        out[col] = pd.to_numeric(out[col].astype(str).str.replace(r"[^\d.]+", "", regex=True), errors="coerce")

    # TEXT: trim spaces and lowercase, so "Ford", "ford" and " FORD " all become "ford".
    # Empty strings and the words "nan"/"none" become a real blank (np.nan).
    for col in ["make", "model"]:
        out[col] = out[col].astype(str).str.strip().str.lower()
        out.loc[out[col].isin(["", "nan", "none"]), col] = np.nan

    # NICKNAMES: one name per brand, so "chevy" and "chevrolet" count as the same make.
    out["make"] = out["make"].replace(NICKNAMES)
    return out


cars = clean_cars(raw)
print(cars["make"].value_counts().head(10))

## 2b. Flag the rows a model should not learn from
Same idea as Lab 3's review flags: an odd price, a placeholder mileage (`999999` means "I don't know"), or a very old
classic car priced by a collector. Flagged rows stay in the data, but they do **not** train the model.

The flag is a small function, so you (and your midterm) can reuse it. Here it decides which cars the model **learns
from**: flagged cars stay in the table but are left out of `model_ready`. Production is different: `predict-shap` still
prices **every** new car, flagged or not (a `$1` ad still has a year, make and model). When you score production later
(the midterm), run this same function first (after renaming `actual_price` to `price`) so junk asking prices do not
count against your model.

So: flagged cars are never learned from, but production still gives them a prediction.

In [ ]:
# The rules, in one place: change a number here and the flag changes everywhere
MIN_PRICE = 500                          # under this: a "$1, call me" bait price or a parts car
MAX_PRICE = 100000                       # over this: usually a typo (an extra zero)
PLACEHOLDER_MILES = [0, 99999, 999999]   # numbers sellers type when they do not know the mileage
OLDEST_YEAR = 1950                       # older than this: a collector car, priced by collectors


def review_reasons(df):
    """One reason string per car: "" means the car looks fine, anything else means "do not learn from me".
    Needs three columns: price, mileage and year. (Production's table calls the price "actual_price":
    rename it first with df.rename(columns={"actual_price": "price"}).)"""
    reasons = []                          # will hold ONE reason string per car, in the same order as the rows of df
    for i in range(len(df)):              # look at the cars one at a time
        row = df.iloc[i]                  # this car (one row of the table)
        why = []                          # this car's list of problems: starts empty, every broken rule adds a label

        # WEIRD PRICE: missing, or outside MIN_PRICE to MAX_PRICE
        if pd.isna(row["price"]) or row["price"] < MIN_PRICE or row["price"] > MAX_PRICE:
            why.append("odd price")

        # WEIRD MILES: a placeholder, not a real odometer reading (see others, like 123456? add them to PLACEHOLDER_MILES)
        if row["mileage"] in PLACEHOLDER_MILES:
            why.append("placeholder mileage")

        # WEIRD YEAR: missing, or older than OLDEST_YEAR (a 1930 Ford is priced by collectors, not by miles and age)
        if pd.isna(row["year"]) or row["year"] < OLDEST_YEAR:
            why.append("very old")

        # Turn this car's list into one string: [] -> "" (a clean car), ["odd price"] -> "odd price",
        # ["odd price", "very old"] -> "odd price, very old". Then add it to the list for all cars.
        reasons.append(", ".join(why))
    return reasons                        # one string per car, so it fits straight into a new column


cars["reason"] = review_reasons(cars)                # new column: "" for clean cars, the problems for flagged ones
cars["needs_review"] = cars["reason"] != ""          # new True/False column: True = at least one problem

print(cars["needs_review"].value_counts())
print()
print("Two new columns on the far right: needs_review (True/False) and reason.")
print("Flagged cars are sorted to the top so you can see them next to normal cars:")
cars.sort_values("needs_review", ascending=False)[["post_id", "year", "make", "model", "price", "mileage", "needs_review", "reason"]].head(10)

Every car is still in `cars`. The flag only decides **which cars the model learns from**: `model_ready` keeps the
rows where `needs_review` is `False`. Here are just the flagged ones, and why. Look at each one: would you trust a
model that learned from it?

In [ ]:
model_ready = cars[cars["needs_review"] == False].copy()
print(len(model_ready), "of", len(cars), "cars are ready for a model |", len(cars) - len(model_ready), "flagged for review")
cars[cars["needs_review"] == True][["post_id", "year", "make", "model", "price", "mileage", "reason"]]

### Disagree with a flag? (optional, a little advanced)
A flag is a rule, and rules make mistakes. Say you open the ad for the 1935 Ford coupe and it really is a restored
classic at $25,000. Put its `post_id` in `KEEP_ANYWAY` and it goes back into training. Its `reason` column still says
"very old", so there is a record of why it was flagged **and** that a person overrode it (good practice on the job).
Use the same list if you score production later (the midterm).

Two warnings: one rare car teaches the model very little (and KNN may let it pull up prices of similar old cars), and
check for duplicates first: two "1930 ford model, $21,500, 500 miles" rows are almost certainly the same car posted twice.

In [ ]:
KEEP_ANYWAY = []     # e.g. ["3mQgZPD6mXiKZnFgdzAZw8"] to keep the 1935 Ford coupe (row 397 above)

if len(KEEP_ANYWAY) > 0:
    cars.loc[cars["post_id"].isin(KEEP_ANYWAY), "needs_review"] = False
    model_ready = cars[cars["needs_review"] == False].copy()
    print("Kept", len(KEEP_ANYWAY), "flagged car(s) on purpose ->", len(model_ready), "cars ready for a model")
else:
    print("No overrides: every flagged car stays out of training.")

# Part 3 · Baseline (10 min)

Before tuning anything, write down the numbers to beat:
1. **Guess the median price for every car.** If your model cannot beat this, it learned nothing.
2. **A plain decision tree** with the same settings as `train-dt`, the model your pipeline already trains every hour.

We hold out 20% of the cars as a **test set** and never tune on them.

## 3a. Split the cars
`X` is the four inputs, `y` is the price. 20% of the cars are locked away as the **test set**; `SEED = 42` makes the split
(and every model's randomness) identical for everyone, so your numbers match your neighbor's. It is saved in the model card.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

SEED = 42                     # one seed for the split AND every model below (saved in the model card)
TEST_SIZE = 0.2               # 20% of the cars become the test set

X = model_ready[FEATURES]
y = model_ready["price"]
# train_test_split gives back FOUR things, always in this order: train inputs, test inputs, train prices, test prices
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(len(X_train), "training cars |", len(X_test), "test cars")

## 3b. The recipe: `make_pipe`
Fill blanks, scale the numbers, one-hot encode make and model, **then** the model, all in one object. You ship that
one object, so production prepares every car exactly the same way.
- *Advanced / more detail:*
  - **Fill blanks:** a missing year or mileage gets the median; a missing make or model gets the most common one
    (learned from training cars only, so no test information leaks in).
  - **Scale:** year (about 2000 to 2025) and mileage (0 to 300,000) end up on the same scale. Trees do not care;
    KNN does, because it measures distances between cars.
  - **One-hot:** one 0/1 column per make and per model. A make seen only once, or never (a Saturn), goes into a shared
    "infrequent" column instead of crashing the model.

In [ ]:
def make_pipe(estimator):
    numeric = Pipeline([("imp", SimpleImputer(strategy="median")),        # year, mileage: fill blanks with the median,
                        ("scale", StandardScaler())])                      # then put both on the same scale
    categorical = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),                       # make, model: fill blanks,
                            ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2))])  # then 0/1 columns
    pre = ColumnTransformer([("num", numeric, ["year", "mileage"]),       # numbers go to the numeric recipe,
                             ("cat", categorical, ["make", "model"])])    # text goes to the categorical one
    return Pipeline([("pre", pre), ("model", estimator)])                # preprocessing, then the model

**Peek inside.** Run just the recipe on your training cars: how many columns the model really sees, which makes are
rare enough (seen once) to share one "infrequent" column (typos like `volkwagen` often land there), and the first 5 cars
as the model sees them (scroll right).

In [ ]:
peek = make_pipe(DecisionTreeRegressor()).named_steps["pre"].fit(X_train)     # the recipe, learned from training cars
columns_seen = peek.get_feature_names_out()
rare_makes = peek.named_transformers_["cat"].named_steps["oh"].infrequent_categories_[0]
print("4 inputs ->", len(columns_seen), "columns. Rare makes (one shared column):", list(rare_makes))

first5 = peek.transform(X_train.head())
if hasattr(first5, "toarray"):
    first5 = first5.toarray()                                                    # sparse (mostly zeros) -> plain table
pd.DataFrame(first5, columns=columns_seen, index=X_train.head().index).round(2)

## 3c. Simple baseline: guess the median
Predict the median training price for **every** test car. If a model cannot beat this, it learned nothing. Notice the
median is well below the mean: a few expensive trucks pull the average up (used-car prices are right-skewed).

In [ ]:
median_price = y_train.median()
print("Median training price: $" + format(round(median_price), ","), "| mean: $" + format(round(y_train.mean()), ","))
median_mae = mean_absolute_error(y_test, np.full(len(y_test), median_price))
print("Guess the median for every test car: test MAE $" + format(round(median_mae), ","))

## 3d. Baseline: a plain decision tree
The same settings as `train-dt`, the tree your pipeline already trains every hour. It gets **two** scores:
- **`train_cv_mae`** uses the **training cars only**. Cross-validation splits them into 5 parts, trains on 4, scores
  on the 5th, and repeats, so every training car gets a turn at being "new". You get 5 MAEs: `train_cv_mae` is their
  **average**, and the `+/-` is their **standard deviation** (a big one means the score depends a lot on which cars
  landed in which fold).
- **`test_mae`** is the error on the held-out test cars, which no model ever learns from or tunes on.

You tune and choose with `train_cv_mae`, and keep the test cars for the very end.

In [ ]:
baseline = make_pipe(DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=SEED))
baseline.fit(X_train, y_train)
baseline_mae = mean_absolute_error(y_test, baseline.predict(X_test))
# scikit-learn always treats a BIGGER score as better, so it reports error as a NEGATIVE number ("neg_" MAE):
# -3,900 beats -4,500. The minus sign on the next line flips it back into a normal, positive MAE in dollars.
baseline_cv_scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
baseline_train_cv_mae = -baseline_cv_scores.mean()      # average MAE over the 5 folds (training cars only)
baseline_train_cv_std = baseline_cv_scores.std()         # how much that MAE wobbles from fold to fold
print("Baseline tree (train-dt): train_cv_mae $" + format(round(baseline_train_cv_mae), ","),
      "+/- $" + format(round(baseline_train_cv_std), ","), "| test MAE $" + format(round(baseline_mae), ","))

# Part 4 · Tune three ways (25 min)

`GridSearchCV` tries every combination in a grid, scores each one with **5-fold cross-validation** on the training
cars (MAE, lower is better), and keeps the best. Three model families, three grids. Run it: it takes a minute or two.

**Want a better model?** Add values to any grid (more `n_neighbors`, other `max_depth` or `min_samples_leaf` values,
more trees) and re-run. **Class challenge:** who gets the lowest `train_cv_mae`? (Bigger grids take longer: every
extra value multiplies the number of models trained.)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

candidates = {
    "decision_tree": (DecisionTreeRegressor(random_state=SEED),
                      {"model__max_depth": [3, 5, 8, 12, None], "model__min_samples_leaf": [1, 5, 10, 20]}),
    "random_forest": (RandomForestRegressor(random_state=SEED, n_jobs=-1),
                      {"model__n_estimators": [100, 300], "model__max_depth": [None, 10],
                       "model__min_samples_leaf": [1, 3, 5]}),
    "knn": (KNeighborsRegressor(),
            {"model__n_neighbors": [3, 5, 7, 11, 15], "model__weights": ["uniform", "distance"]}),
}

searches = {}
rows = []
for name in candidates:
    estimator, grid = candidates[name]
    search = GridSearchCV(make_pipe(estimator), grid, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
    search.fit(X_train, y_train)
    searches[name] = search
    test_mae = mean_absolute_error(y_test, search.best_estimator_.predict(X_test))
    best = search.best_index_
    rows.append({"model": name,
                 "train_cv_mae": round(-search.best_score_),           # minus: flip sklearn's negative MAE back to dollars
                 "train_cv_mae_std": round(search.cv_results_["std_test_score"][best]),
                 "test_mae": round(test_mae),
                 "best_params": search.best_params_})

rows.append({"model": "baseline tree (not tuned)", "train_cv_mae": round(baseline_train_cv_mae),
             "train_cv_mae_std": round(baseline_train_cv_std), "test_mae": round(baseline_mae),
             "best_params": "max_depth=12, min_samples_leaf=10"})
results = pd.DataFrame(rows).sort_values("train_cv_mae").reset_index(drop=True)
results

## Your champion
The cell below **picks the champion for you**: the tuned model with the lowest `train_cv_mae`. It never looks at
`test_mae`, because the test set is your final exam and you do not choose with it.

Want to ship a different model (say, KNN, to see how its SHAP explanations look in production)? **Uncomment the second
line** and type its name: `decision_tree`, `random_forest` or `knn`.

**Why do `train_cv_mae` and `test_mae` disagree?** The test set is small, so a few unusual cars can move its MAE by
hundreds of dollars. `train_cv_mae` averages five different splits, so it is the steadier number (and its `std`
tells you how steady). A model can even look worse than the baseline on one and better on the other. That is small
data, not a bug.

In [ ]:
# The results table also lists the baseline tree. It is the number to beat, not a candidate, so drop it first.
tuned_models = results[results["model"] != "baseline tree (not tuned)"]
CHAMPION = tuned_models.iloc[0]["model"]        # the table is sorted by train_cv_mae, so the first row is the champion
# CHAMPION = "knn"                              # <-- uncomment to override: "decision_tree", "random_forest" or "knn"

champion = searches[CHAMPION].best_estimator_
champion_train_cv_mae = -searches[CHAMPION].best_score_
champion_train_cv_std = searches[CHAMPION].cv_results_["std_test_score"][searches[CHAMPION].best_index_]
champion_test_mae = mean_absolute_error(y_test, champion.predict(X_test))
print("Champion:", CHAMPION, searches[CHAMPION].best_params_)
print("Champion: train_cv_mae $" + format(round(champion_train_cv_mae), ","), "+/- $" + format(round(champion_train_cv_std), ","),
      "| test MAE $" + format(round(champion_test_mae), ","))
print("Baseline: train_cv_mae $" + format(round(baseline_train_cv_mae), ","), "+/- $" + format(round(baseline_train_cv_std), ","),
      "| test MAE $" + format(round(baseline_mae), ","))

## Your turn: is your champion really better? (15 min, you write the code)
Numbers like MAE hide a lot. Before you ship, **look** at how your champion and the baseline tree miss, car by car,
on the test cars. Run the setup cell, then write the three short pieces of code yourself (ask Claude if you get stuck,
but type it in yourself). Error = predicted price minus asking price, so **positive = the model priced the car too high**.

In [ ]:
# Setup (given): predictions and errors on the test cars, for your champion and for the baseline tree
champ_pred = champion.predict(X_test)
base_pred = baseline.predict(X_test)
champ_err = champ_pred - y_test        # positive = the model said MORE than the asking price
base_err = base_pred - y_test
print("Ready:", len(y_test), "test cars | champ_pred, base_pred, champ_err, base_err")

**Task 1: actual vs predicted.** Two scatter plots side by side (champion, then baseline). Asking price on the x-axis,
predicted price on the y-axis, and a dashed diagonal line where predicted = actual. Dots on the line are perfect.

*Hints:* `fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)`, then `axes[0].scatter(y_test, champ_pred, alpha=0.6)`,
and `axes[0].plot([0, top], [0, top], "--")` with `top = y_test.max()`. Give each plot a title and axis labels.

In [ ]:
# YOUR CODE HERE: two scatter plots, actual (x) vs predicted (y), champion on the left, baseline on the right

**Task 2: how big are the misses?** A histogram of `champ_err` and `base_err` on the same plot (use `alpha=0.5` so
both show), with 30 bins and a vertical line at 0. A tall, narrow pile around 0 is what you want.

*Hints:* `plt.hist(champ_err, bins=30, alpha=0.5, label="champion")`, the same for `base_err`, then
`plt.axvline(0, color="black")`, `plt.legend()`, axis labels, `plt.show()`.

In [ ]:
# YOUR CODE HERE: overlaid histograms of champ_err and base_err, 30 bins, a vertical line at 0

**Task 3: summary stats.** A small table with one column per model and these rows: mean error (bias: does it price
too high or too low on average?), MAE, median absolute error, standard deviation of the error, and the largest miss.

*Hints:* build a dictionary per model, e.g. `{"mean error": champ_err.mean(), "MAE": champ_err.abs().mean(), ...}`, then
`pd.DataFrame({"champion": champ_stats, "baseline": base_stats}).round(0)`.

In [ ]:
# YOUR CODE HERE: a table of mean error, MAE, median absolute error, std of error and largest miss, for both models

**Talk it over with a neighbor (and write one sentence in Part 8):** where does your champion beat the baseline:
cheap cars, expensive cars, everywhere? Where do both still miss badly, and what do those cars have in common?

# Part 5 · Explain it before you ship it (20 min)

Lab 2 came back. SHAP splits each prediction into **one contribution per feature**, measured from the average
prediction (the **base value**): `base value + SHAP(year) + SHAP(mileage) + SHAP(make) + SHAP(model) = prediction`.

The two helpers below are the **same code `predict-shap` runs in production**. Trees get exact, fast SHAP (`TreeExplainer`);
KNN gets a slower, sampled estimate (`KernelExplainer`). The one-hot columns (98 on Dr. Wanik's cars, see Part 3b) are added back up, so you see 4 features.

In [ ]:
# HELPER 1 of 2 (same code as predict-shap in production). Run it; you do not need to edit it.
# The pipeline one-hot encodes make and model, so the model really sees ~100 columns (see Part 3b) with names like
# "cat__make_honda" or "cat__model_civic". SHAP gives one value per COLUMN. This function tells us which of
# our 4 original features each column came from, so we can add the pieces back up into year/mileage/make/model.

def original_feature(transformed_name):
    name = transformed_name.split("__", 1)[-1]        # "cat__make_honda" -> "make_honda"
    for col in ["make", "model"]:
        if name.startswith(col + "_"):                 # "make_honda" -> "make"
            return col
    for col in ["year", "mileage"]:
        if name == col or name.startswith(col):        # "year" -> "year"
            return col
    return name

**Helper 2: `explain_rows`.** It returns **two things**: the **base value** (the model's average prediction,
where every car starts) and a **table** with one row per car and one SHAP column per feature.

**If your model is made of trees, SHAP reads the trees directly: exact and fast. If not (KNN), SHAP has to experiment
with what-if versions of each car: approximate and slow.**

(For the curious: the what-if versions swap in values from "typical cars", and `shap.kmeans` picks 10 typical cars so
the experiment stays fast. The next cell times SHAP for your champion.)

In [ ]:
# HELPER 2 of 2 (same code as predict-shap in production). Run it; you do not need to edit it.

def explain_rows(pipe, X, background):
    pre = pipe.named_steps["pre"]                      # the preprocessing half of your pipeline
    est = pipe.named_steps["model"]                    # the model half (tree, forest or KNN)
    Xt = pre.transform(X)                              # the cars as the model sees them (~100 columns)
    if hasattr(Xt, "toarray"):
        Xt = Xt.toarray()                              # one-hot output can be "sparse"; SHAP wants a plain table
    names = list(pre.get_feature_names_out())          # the ~100 column names, e.g. "cat__make_honda"

    if hasattr(est, "tree_") or hasattr(est, "estimators_"):
        # TREE or FOREST: exact SHAP, straight from the trees
        explainer = shap.TreeExplainer(est)
        values = explainer.shap_values(Xt, check_additivity=False)
        base = float(np.ravel(explainer.expected_value)[0])
    else:
        # KNN (or any other model): estimated SHAP, swapping in values from 10 typical cars
        Bt = pre.transform(background)
        if hasattr(Bt, "toarray"):
            Bt = Bt.toarray()
        summary = shap.kmeans(Bt, min(10, len(Bt)))   # 10 representative "typical cars" (cluster centers)
        explainer = shap.KernelExplainer(est.predict, summary)
        values = explainer.shap_values(Xt, nsamples=100, silent=True)   # 100 what-if samples per car
        base = float(np.ravel(explainer.expected_value)[0])
    values = np.asarray(values)

    # Add the ~100 column-level SHAP values back up into our 4 features
    shap_columns = []
    for feature in FEATURES:
        shap_columns.append("shap_" + feature)
    grouped = pd.DataFrame(0.0, index=X.index, columns=shap_columns)
    for j in range(len(names)):
        grouped["shap_" + original_feature(names[j])] += values[:, j]
    return base, grouped                               # TWO things: the base value, and the SHAP table

Run the next cell to get SHAP values for every test car. You get the **base value** (where every car starts) and how far
the actual predictions spread around it. For every car, base value + its four SHAP values = its prediction, exactly.

In [ ]:
background = X_train.sample(min(50, len(X_train)), random_state=0)
import time
shap_start = time.time()
shap_result = explain_rows(champion, X_test, background)    # gives back TWO things...
base_value = shap_result[0]                                # ...1. the base value (one number)
test_shap = shap_result[1]                                 # ...2. a table: one row per test car, one SHAP column per feature
test_preds = champion.predict(X_test)
print("SHAP took", round(time.time() - shap_start, 1), "seconds for", len(X_test), "cars with your", CHAMPION,
      "(trees: TreeExplainer, exact and fast; KNN: KernelExplainer, sampled and much slower)")
print("Base value (the model's AVERAGE prediction, where every car starts): $" + format(round(base_value), ","))
print("Its actual predictions on the test cars run from $" + format(round(test_preds.min()), ","),
      "to $" + format(round(test_preds.max()), ","), "(spread, std: $" + format(round(test_preds.std()), ",") + ")")

**Which features drive your model's prices?** Mean absolute SHAP = how many dollars each feature moves a typical prediction.

In [ ]:
importance = test_shap.abs().mean().sort_values()
importance.index = importance.index.str.replace("shap_", "")
importance.plot(kind="barh", color="#6D28A3")
plt.xlabel("Mean |SHAP| (dollars)")
plt.title("What drives " + CHAMPION + "'s prices (test cars)")
plt.show()

**The global picture: a beeswarm.** Every dot is one test car. Left of the center line, that feature pushed the car's
price **down**; right, **up**. Color is the feature's value: red = high, blue = low (make and model are text, so they are
gray). Look at mileage: red dots (high miles) on the left means more miles, lower price. Is that what you expected?

In [ ]:
# A SHAP "Explanation" bundles the SHAP values with each car's real values, so the plot can color the dots
beeswarm_data = X_test[["year", "mileage"]].copy()
beeswarm_data["make"] = np.nan          # text columns cannot be colored high/low, so they show as gray
beeswarm_data["model"] = np.nan
everyone = shap.Explanation(values=test_shap.values,
                            base_values=np.full(len(test_shap), base_value),
                            data=beeswarm_data.values.astype(float),
                            feature_names=FEATURES)
shap.plots.beeswarm(everyone, show=False)
plt.title("How each feature pushed every test car's price (" + CHAMPION + ")")
plt.show()

**One car, explained.** The waterfall starts at the base value and adds each feature's push until it reaches the
prediction. Change `CAR` to look at another test car.

In [ ]:
def waterfall(base, shap_row, feature_row):
    explanation = shap.Explanation(values=np.array(shap_row, dtype=float), base_values=base,
                                   data=np.array(feature_row, dtype=object), feature_names=FEATURES)
    shap.plots.waterfall(explanation, show=True)


CAR = 0
car_features = X_test.iloc[CAR]
print("Actual price: $" + format(round(y_test.iloc[CAR]), ","), "| predicted: $" + format(round(champion.predict(X_test.iloc[[CAR]])[0]), ","))
waterfall(base_value, test_shap.iloc[CAR].values, car_features.values)

## 5b. Price a car you know
Your car, a parent's car, the car you wish you had. Type it in, and your champion prices it and explains itself.
Write the make and model the way Craigslist sellers do (`honda`, `civic`).

If the model has never seen that make or model, it treats it as "a rare car" and leans on year and mileage alone.
The cell tells you how many it trained on.

In [ ]:
MY_YEAR = 2015
MY_MILEAGE = 90000
MY_MAKE = "honda"
MY_MODEL = "civic"

# Build a one-row table that looks exactly like a scraped car. Price is unknown (NaN): that is what we predict.
my_car = pd.DataFrame([{"price": np.nan, "year": MY_YEAR, "mileage": MY_MILEAGE, "make": MY_MAKE, "model": MY_MODEL}])

# Clean it the SAME way as the training cars (lowercase, "chevy" -> "chevrolet", numbers as numbers),
# then keep only the 4 columns the model takes as input. Without this, "Honda" and "honda" would be different makes.
my_car = clean_cars(my_car)[FEATURES]

# Ask the champion for a price, and ask SHAP why.
my_price = champion.predict(my_car)[0]
my_result = explain_rows(champion, my_car, background)      # TWO things again:
my_base = my_result[0]                                     # the base value
my_shap = my_result[1]                                     # the SHAP table (one row: your car)

# How many training cars had the same make / model? Few or zero means the model is mostly guessing from year and miles.
same_make = int((X_train["make"] == my_car.iloc[0]["make"]).sum())
same_model = int((X_train["model"] == my_car.iloc[0]["model"]).sum())
print("Your model trained on", same_make, "cars with make =", my_car.iloc[0]["make"],
      "and", same_model, "with model =", my_car.iloc[0]["model"])
if same_make == 0:
    print("  ->", my_car.iloc[0]["make"], "was never seen in training, so the model treats it as a RARE make (the 'infrequent' group)")
if same_model == 0:
    print("  ->", my_car.iloc[0]["model"], "was never seen in training, so the model treats it as a RARE model (the 'infrequent' group)")
print("Your model's price: $" + format(round(my_price), ","))

# The waterfall: start at the base value (bottom), each feature pushes the price up (red) or down (blue).
waterfall(my_base, my_shap.iloc[0].values, my_car.iloc[0].values)

# Part 6 · Ship it (20 min)

## 6a. Save the model and its model card
`model.joblib` is your whole pipeline, frozen. `model_card.json` is its label: what it is, how good it was, which
library versions made it, and which cars it trained on (so production can tell **new** cars from cars it has seen).

In [ ]:
from datetime import datetime, timezone

joblib.dump(champion, "/content/model.joblib")
best_params = {}
for key in searches[CHAMPION].best_params_:
    best_params[key] = str(searches[CHAMPION].best_params_[key])
card = {
    "model_type": type(champion.named_steps["model"]).__name__,
    "champion": CHAMPION,
    "best_params": best_params,
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "features": FEATURES,
    "train_cv_mae": round(float(champion_train_cv_mae), 2),
    "train_cv_mae_std": round(float(champion_train_cv_std), 2),
    "test_mae": round(float(champion_test_mae), 2),
    "baseline_mae": round(float(baseline_mae), 2),
    "seed": SEED,                                   # how to rebuild this exact model: same seed...
    "test_size": TEST_SIZE,                         # ...same split...
    "cv_folds": 5,                                  # ...same cross-validation...
    "n_train": len(X_train),
    "n_test": len(X_test),
    "data_source": "own bucket" if USE_MY_OWN_DATA else "Dr. Wanik's snapshot (" + str(len(raw)) + " cars)",
    "train_post_ids": list(model_ready.loc[X_train.index, "post_id"].astype(str)),
}
with open("/content/model_card.json", "w") as f:
    json.dump(card, f, indent=2)
print("Saved /content/model.joblib and /content/model_card.json (see the Files panel on the left)")
card_to_show = dict(card)
card_to_show.pop("train_post_ids")
print(json.dumps(card_to_show, indent=2))

## 6b. Sign in and check (3 checks, all in Colab)
This uses the `PROJECT_ID` and `BUCKET_NAME` you typed in Part 1b. When the **"Allow access"** pop-up opens, pick the
**Gmail you used for Google Cloud**. That choice matters: Colab talks to Google Cloud as whichever account you pick,
and any other account gets "403 Forbidden". The cell then checks, in plain English:
1. **who** you signed in as,
2. that this account can see **your bucket**,
3. that **`predict-shap`** is deployed and ACTIVE (from Part 0).

No Google Cloud yet? Set `UPLOAD = False`: you can still do Part 7 in demo mode.

In [ ]:
UPLOAD = True

if UPLOAD and ("YOUR-PROJECT-ID" in PROJECT_ID or "YOUR-PROJECT-ID" in BUCKET_NAME):
    print("Type your PROJECT_ID and BUCKET_NAME in Part 1b, re-run that cell, then re-run this one.")
    print("Staying in demo mode until you do.")
    UPLOAD = False

if UPLOAD:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()

    # CHECK 1: which Google account did you just pick in the pop-up?
    account_lines = !gcloud config get-value account
    signed_in_as = "unknown"
    if len(account_lines) > 0:
        signed_in_as = account_lines[-1].strip()
    print("1. Signed in as:", signed_in_as)

    # CHECK 2: can that account see your bucket?
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    try:
        bucket_ok = bucket.exists()
    except Exception as err:
        bucket_ok = False
        print("   Google says:", str(err)[:200])
    if not bucket_ok:
        raise RuntimeError("The account " + signed_in_as + " cannot see the bucket " + BUCKET_NAME + ". "
                           "Check BUCKET_NAME in Part 1b, or Runtime > Disconnect and delete runtime, then run again "
                           "and pick your Google Cloud Gmail in the pop-up.")
    print("2. Bucket found:", BUCKET_NAME)

    # CHECK 3: is predict-shap deployed and running?
    state_lines = !gcloud functions describe predict-shap --gen2 --region=us-central1 --project={PROJECT_ID} --format="value(state)"
    state = "NOT FOUND"
    if len(state_lines) > 0:
        state = state_lines[-1].strip()
    if state != "ACTIVE":
        raise RuntimeError("predict-shap is not ACTIVE (Google says: " + state[:150] + "). Go back to Part 0: "
                           "case A or B, look for a red X in your repo's Actions tab; case C, re-run the Cloud Shell block.")
    print("3. predict-shap is ACTIVE. Ready to ship.")

## 6c. Upload your model to your bucket
Two copies: `models/model.joblib` is the **live** model `predict-shap` uses (it is replaced every time you ship), and
`models/history/model_<time>.joblib` is a copy **named by time** that is never overwritten, so every model you ever
shipped is kept.

In [ ]:
if UPLOAD:
    bucket.blob("models/model.joblib").upload_from_filename("/content/model.joblib")          # the LIVE model
    bucket.blob("models/model_card.json").upload_from_filename("/content/model_card.json")
    stamp = card["trained_at"].replace(":", "-")                                               # e.g. 2026-10-13T14-30-00Z
    bucket.blob("models/history/model_" + stamp + ".joblib").upload_from_filename("/content/model.joblib")    # named by time,
    bucket.blob("models/history/model_card_" + stamp + ".json").upload_from_filename("/content/model_card.json")  # never overwritten
    print("Live model:  gs://" + BUCKET_NAME + "/models/model.joblib")
    print("Kept a copy: gs://" + BUCKET_NAME + "/models/history/model_" + stamp + ".joblib")
    print("predict-shap uses it at the next :25, or right now in 6d")
else:
    print("UPLOAD is False: nothing uploaded. Part 7 will run in demo mode.")

## 6d. Run `predict-shap` now
Normally `predict-shap` runs by itself every hour at :25. This cell starts it **right now** from Colab (the account you
picked in 6b works for Google Cloud commands here), then checks your bucket every 15 seconds until it finishes
(usually under a minute; KNN takes longer).

**What `predict-shap` does each time it runs** (the code is `cloud_function/predict-shap/main.py` in your fork):
1. **Loads your model**: `models/model.joblib`, plus `model_card.json`, whose `trained_at` is the model's version stamp.
2. **Loads every car** in `listings_master.csv` and cleans them with the same `clean_cars` you used in Part 2.
3. **Finds the cars this model version has not priced yet.** The **first run after you upload** (this one) finds
   **all of them**, so it prices your whole table. **Every hourly run after that** finds only the new listings the
   scraper added that hour (usually 10 or fewer). Only **one** model is ever live: uploading a new one stops the old
   one. Each run prices at most 500 cars (40 for KNN), so a table bigger than that takes more than one run to catch up.
4. **Predicts** a price for each of those cars, and **explains** each one with SHAP (the same `explain_rows` as Part 5).
5. **Appends** (never overwrites) one row per car to two files in your bucket:
   - `gs://YOUR-BUCKET/structured/preds_lab4/preds_master.csv`: the car, asking price, predicted price, error,
     `seen_in_training`, model version, time
   - `gs://YOUR-BUCKET/structured/preds_lab4/shap_master.csv`: the base value and the 4 SHAP values behind each
     price, model version, time

Because every row is tagged with the model version, uploading a new model later keeps the old rows: the new version
just prices everything again next to them.

In [ ]:
import time
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

if UPLOAD:
    started = datetime.now(timezone.utc)
    !gcloud scheduler jobs run predict-shap-hourly --location=us-central1 --project={PROJECT_ID}
    print("Started predict-shap. Waiting for it to write your predictions", end="")
    preds_blob = bucket.blob("structured/preds_lab4/preds_master.csv")
    ready = False
    for attempt in range(16):                 # check every 15 seconds, for up to 4 minutes
        time.sleep(15)
        print(".", end="")
        if preds_blob.exists():
            preds_blob.reload()
            if preds_blob.updated > started:  # the file changed after we started: predict-shap is done
                ready = True
                break
    print()
    if ready:
        done_at = preds_blob.updated.astimezone(ZoneInfo("America/New_York")).strftime("%I:%M %p")
        print("Done: predict-shap wrote your predictions at", done_at, "(Eastern). Go to Part 7.")
        print("  gs://" + BUCKET_NAME + "/structured/preds_lab4/preds_master.csv")
        print("  gs://" + BUCKET_NAME + "/structured/preds_lab4/shap_master.csv")
    else:
        print("Nothing new after 4 minutes. Run the optional log cell below and paste the log into Claude.")
else:
    print("UPLOAD is False: skipping. Part 7 will run in demo mode.")

**Optional, only for troubleshooting.** Skip this if 6d said "wrote new predictions". If it did not, this cell
prints the last 20 lines of `predict-shap`'s log. `"status": "no_model"` means the upload
in 6c did not run; anything with `Error` is worth pasting into Claude. (Prefer Cloud Shell? The same two commands work
there: `gcloud scheduler jobs run predict-shap-hourly --location=us-central1` and the logs command below.)

In [ ]:
if UPLOAD:
    !gcloud functions logs read predict-shap --region=us-central1 --gen2 --limit=20 --project={PROJECT_ID}
else:
    print("UPLOAD is False: no log to read.")

# Part 7 · Your frozen model at work (5 min)
Your model never retrains on its own: every hour it prices the new cars and saves each price with its SHAP values.
Here is what it saved.

## 7a. Load production (plumbing: just run it)
Reads the two files `predict-shap` writes in your bucket:
- **`gs://YOUR-BUCKET/structured/preds_lab4/preds_master.csv`**: each car, its asking price and your model's price
- **`gs://YOUR-BUCKET/structured/preds_lab4/shap_master.csv`**: the base value and the 4 SHAP values behind each price
No upload in Part 6? It builds demo tables instead.

In [ ]:
# PLUMBING (run me, no need to read): load production's two files, or build demo tables if there are none yet.


def read_bucket_csv(key):
    blob = bucket.blob(key)
    if not blob.exists():
        return None
    return pd.read_csv(io.BytesIO(blob.download_as_bytes()), dtype={"post_id": str, "model_version": str})


def demo_tables():
    seen_cars = cars.loc[X_train.index].sample(50, random_state=1)
    demo_cars = pd.concat([seen_cars, cars.loc[X_test.index], cars[cars["needs_review"]]])
    demo_preds = demo_cars[["post_id", "year", "mileage", "make", "model"]].copy()
    demo_preds["actual_price"] = demo_cars["price"]
    demo_preds["pred_price"] = champion.predict(demo_cars[FEATURES]).round(2)
    demo_preds["abs_error"] = (demo_preds["pred_price"] - demo_preds["actual_price"]).abs()
    demo_preds["seen_in_training"] = demo_cars.index.isin(X_train.index)
    demo_result = explain_rows(champion, demo_cars[FEATURES], background)
    demo_base = demo_result[0]
    demo_shap = demo_result[1]
    demo_shap["post_id"] = demo_cars["post_id"]
    demo_shap["base_value"] = demo_base
    demo_shap["pred_price"] = demo_preds["pred_price"]
    return demo_preds, demo_shap


IN_PRODUCTION = False      # becomes True only if production has really priced cars with your current model
if UPLOAD:
    live_version = str(json.loads(bucket.blob("models/model_card.json").download_as_text())["trained_at"])
    preds = read_bucket_csv("structured/preds_lab4/preds_master.csv")
    prod_shap = read_bucket_csv("structured/preds_lab4/shap_master.csv")
    if preds is not None and prod_shap is not None:
        preds = preds[preds["model_version"] == live_version].copy()
        prod_shap = prod_shap[prod_shap["model_version"] == live_version].copy()
        IN_PRODUCTION = len(preds) > 0
if not IN_PRODUCTION:
    print("DEMO MODE (no production predictions from your model yet: run 6d, then re-run this cell)")
    preds, prod_shap = demo_tables()

if IN_PRODUCTION:
    print("Loaded", len(preds), "predictions and their SHAP values from production. Go to 7b.")
else:
    print("Loaded", len(preds), "DEMO predictions (built here, not from production). Go to 7b.")

## 7b. What your model priced, and why
The newest cars your model never saw: its price, and the SHAP values behind it
(`base_value` + the 4 SHAP values = `pred_price`). Click the link to see the same rows in Google Cloud.

**What to expect tonight:** you trained on Dr. Wanik's Hartford cars, but production prices **your own** bucket's cars
(your Craigslist area), so all of them are brand new to your model (`seen_in_training = False`): a real test of how a
Hartford model does in your market. Production prices at most **500 cars per run** (40 for KNN). If the first line
below says exactly 500, you have more cars than that: the rest get priced at the next :25 runs. Nothing is broken.

In [ ]:
print("Your frozen model has priced", len(preds), "cars;", int((preds["seen_in_training"] == False).sum()), "of them it never saw in training.")
if IN_PRODUCTION:
    print("Your two production files:")
    print("  gs://" + BUCKET_NAME + "/structured/preds_lab4/preds_master.csv")
    print("  gs://" + BUCKET_NAME + "/structured/preds_lab4/shap_master.csv")
    print("Open that folder in Google Cloud: https://console.cloud.google.com/storage/browser/" + BUCKET_NAME
          + "/structured/preds_lab4?project=" + PROJECT_ID)

new_cars = preds[preds["seen_in_training"] == False]                     # brand new to your model
shap_cols = ["post_id", "base_value", "shap_year", "shap_mileage", "shap_make", "shap_model"]
new_cars = new_cars.merge(prod_shap[shap_cols], on="post_id", how="left")  # attach production's SHAP values
new_cars[["year", "make", "model", "mileage", "actual_price", "pred_price",
          "base_value", "shap_year", "shap_mileage", "shap_make", "shap_model"]].tail(10).round(0)

# Part 8 · Wrap up (5 min)

## Five short answers (1 or 2 sentences each)
1. Which model family won on `train_cv_mae`, and by how many dollars did it beat the baseline tree? Was its `std` small or large?
2. The car you priced in 5b: was the price believable? Which feature pushed it the most, and in which direction?
3. Part 7b: pick one new car. Which feature pushed its price the most, and does that match the ad? Then open
   `preds_master.csv` in Google Cloud: how many different `model_version` values does it have, and why?
4. Part 5b: the car you priced. Does the waterfall make sense? What would you want the model to know that the four
   features cannot tell it?
5. Your turn: where does your champion beat the baseline, and where do both still miss badly?

*Your answers:*

---

---

## Want a custom model on your own cars? (optional, any time after tonight)
Tonight everyone trained on Dr. Wanik's Hartford cars, so everyone succeeds. Your pipeline keeps collecting **your**
area's cars every hour, so later you can train a model that knows your market:
1. **Check your data first** (Cloud Shell). A06 fixed? Enough cars? Count the makes and the rows:
   ```bash
   gcloud storage cat "gs://YOUR-BUCKET/structured/datasets/listings_master.csv" | cut -d, -f6 | sort | uniq -c | sort -rn | head
   gcloud storage cat "gs://YOUR-BUCKET/structured/datasets/listings_master.csv" | wc -l
   ```
   No `contact` at the top and 100+ rows? You are ready.
2. **Re-run this notebook** with `USE_MY_OWN_DATA = True` in Part 1b. Everything else stays the same: clean, baseline,
   tune, explain, ship.
3. **Ship it** (Part 6c uploads the new `model.joblib` over the live one, and keeps a copy named by time in `models/history/`).
   `predict-shap` notices the new model version and re-predicts and re-explains all your cars with it. The old
   predictions stay in the files, tagged with the old version, so you can compare the two models. Now production prices **your** market.
4. Want a copy of your data on your laptop? `gcloud storage cp "gs://YOUR-BUCKET/structured/datasets/listings_master.csv" .`
   in Cloud Shell, then **⋮ → Download** (Cloud Shell's menu).

## Try it yourself (optional): make and model as ONE column
Some model names belong to more than one make: `1500` is a Chevy, a Ram and a GMC; `grand` is a Jeep Grand Cherokee
and a Dodge Grand Caravan. With separate columns, your model sees one "1500" column for all of them. A combined column
tells them apart:
```python
cars["make_model"] = cars["make"] + " " + cars["model"]      # "ram 1500", "jeep grand", ...
FEATURES = ["year", "mileage", "make", "make_model"]          # keep make too, so rare pairs still get brand credit
```
Then update `make_pipe` to one-hot `["make", "make_model"]`, re-run from Part 3, and compare `train_cv_mae`. Watch the
"infrequent" list in 3b grow: many make-model pairs appear only once. **Colab only** unless you also teach your own
`predict-shap` the new column (the same three edits as the midterm: `FEATURES`, `clean_cars`, `original_feature`).

**Submit:** File → Download → `.ipynb`, then upload it to **Submit Lab 4** on HuskyCT.
**Leave `predict-shap` running.** Every hour it predicts and explains the newest cars, and your midterm can chart how
your model's errors and explanations move over time.